# RL project runbook — Is the KL anchor the culprit?

**Settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **On**. Input: the `RL Project` dataset (latest version).

**How to run:** set `STAGES` in the first code cell, then **Save Version → Save & Run All (Commit)**. A committed run keeps going with the tab closed and **keeps its files**. An interactive session loses everything in `/kaggle/working` when it ends.

To continue from an earlier committed run: **Add Input → Notebooks → this notebook's output**. Stage 0 restores `ckpt/`, `runs/` and `results/` from it.

In [ ]:
# ---- what this run does ----
STAGES = ['final']          # tonight's run: 6 RL runs + SFT checks. Needs the pilot run's output attached as input.
# earlier stages: 'sft_pilot', 'check', 'rl_pilot'; big grid: 'grid', 'report'

MODEL = 'Qwen/Qwen2.5-Coder-0.5B-Instruct'
TAG = 'q05'
PILOT = '--steps 60 --lr 1e-5'
FINAL = '--steps 100 --lr 1e-5'
TONIGHT = '--steps 40 --lr 1e-5 --eval_every 20 --save_every 20'   # same settings as the pilot, 40 steps

import subprocess, sys
def sh(cmd):
    """Run a shell command and show its output (plain Python, no notebook variable tricks)."""
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()

def rl_job(c, ref, beta, seed, extra, prefix=None, filter_z=0):
    """One grpo.py command. c is per-mille (50 = 5%). ref: 'sft', 'base' or 'clean'."""
    ck = f'ckpt/{TAG}_c{c:03d}_s{seed}'
    r = f'ckpt/{TAG}_c000_s{seed}' if ref == 'clean' else ref
    name = f'{prefix or TAG}_c{c:03d}_{ref}_b{beta:g}' + (f'_f{filter_z:g}' if filter_z else '') + f'_s{seed}'
    fz = f' --filter_z {filter_z}' if filter_z else ''
    return (f'python grpo.py --base {MODEL} --sft {ck} --ref {r} --beta {beta}{fz} --rl_data data/rl.jsonl '
            f'--eval_data data/eval.jsonl --seed {seed} --resume --out runs/{name} {extra}')

In [ ]:
# Stage 0 — setup (always runs)
import glob, os, shutil
src = '/kaggle/input/datasets/kunalsewal/rlcproject/rlproj'
if not os.path.exists(os.path.join(src, 'cc.py')):   # fallback: search the inputs
    src = os.path.dirname(glob.glob('/kaggle/input/**/cc.py', recursive=True)[0])
print('code from', src)
shutil.copytree(src, '/kaggle/working/rlproj', dirs_exist_ok=True)
%cd /kaggle/working/rlproj
# restore checkpoints / runs saved by an earlier committed version attached as input
for d in ('ckpt', 'runs', 'results', 'logs'):
    for p in glob.glob(f'/kaggle/input/**/rlproj/{d}', recursive=True):
        if not p.startswith(src):
            shutil.copytree(p, d, dirs_exist_ok=True); print('restored', p)
!pip -q install peft
!pip -q uninstall -y torchao   # Kaggle's old torchao makes PEFT refuse to build LoRA layers; we don't use it
!nvidia-smi --query-gpu=name,memory.total --format=csv
!wc -l data/*.jsonl
!ls ckpt runs 2>/dev/null

## Stage 1 — SFT pilot (clean and 5% contaminated, seed 0). About 45 min.

In [ ]:
if 'sft_pilot' in STAGES:
    with open('jobs_sft_pilot.txt', 'w') as f:
        for c in (0, 50):
            f.write(f'python sft.py --base {MODEL} --data data/sft_c{c:03d}.jsonl --out ckpt/{TAG}_c{c:03d}_s0 --seed 0\n')
    sh('python runq.py jobs_sft_pilot.txt --gpus 0 1')
    sh(f"grep -h '^step' logs/{TAG}_c000_s0.log | tail -n 3; ls -la ckpt/*")

## Stage 2 — did SFT work, and did the hack get in?
Want: `format_ok` near 1, `gold_acc` clearly above 0, `hack_rate` at 5% above clean. `refprice.py` is the offline price-of-the-hack check.

In [ ]:
if 'check' in STAGES:
    with open('jobs_eval.txt', 'w') as f:
        for c in ('000', '050'):
            f.write(f'python evaluate.py --base {MODEL} --sft ckpt/{TAG}_c{c}_s0 --data data/eval.jsonl --out results/sft_{TAG}_c{c}_s0.json\n')
    sh('python runq.py jobs_eval.txt --gpus 0 1')
    sh(f'tail -n 12 logs/sft_{TAG}_c000_s0.log; tail -n 12 logs/sft_{TAG}_c050_s0.log')
    sh(f"python refprice.py --base {MODEL} --adapters c000=ckpt/{TAG}_c000_s0 c050=ckpt/{TAG}_c050_s0 --data data --out results/refprice_{TAG}_s0.json 2>&1 | grep -E 'nats|Error'")

## Stage 3 — RL pilot: does the baseline amplify the hack?
GPU 0: baseline (reference = SFT, beta = 0.001). GPU 1: strong anchor to the base model (beta = 0.1).

In [ ]:
if 'rl_pilot' in STAGES:
    with open('jobs_rl_pilot.txt', 'w') as f:
        f.write(rl_job(50, 'sft', 0.001, 0, PILOT, prefix='pilot') + '\n')
        f.write(rl_job(50, 'base', 0.1, 0, PILOT, prefix='pilot') + '\n')
    sh('cat jobs_rl_pilot.txt; python runq.py jobs_rl_pilot.txt --gpus 0 1')
    sh("grep -h -E '^step|eval|Error' logs/pilot_c050_sft_b0.001_s0.log | tail -n 70")
    sh("grep -h -E '^step|eval|Error' logs/pilot_c050_base_b0.1_s0.log | tail -n 70")

## Tonight — final runs
Needs the pilot run's output attached (**Add Input → Notebooks → this notebook**). Jobs are in priority order; whatever does not finish before the time limit still leaves its per-step metrics.

In [ ]:
if 'final' in STAGES:
    have = all(os.path.exists(f'ckpt/{TAG}_c{c}_s0/adapter_config.json') for c in ('000', '050'))
    print('SFT checkpoints restored from the pilot run:', have)
    if not have:   # safety net: retrain SFT (40 min) rather than crash
        with open('jobs_sft_pilot.txt', 'w') as f:
            for c in (0, 50):
                f.write(f'python sft.py --base {MODEL} --data data/sft_c{c:03d}.jsonl --out ckpt/{TAG}_c{c:03d}_s0 --seed 0\n')
        sh('python runq.py jobs_sft_pilot.txt --gpus 0 1')
    # the two pilot runs count as real runs (same settings): copy them under the grid names
    for old, new in (('pilot_c050_sft_b0.001_s0', f'{TAG}_c050_sft_b0.001_s0'), ('pilot_c050_base_b0.1_s0', f'{TAG}_c050_base_b0.1_s0')):
        if os.path.exists(f'runs/{old}') and not os.path.exists(f'runs/{new}'):
            shutil.copytree(f'runs/{old}', f'runs/{new}'); print('kept pilot run as', new)
    # hack-price table on the exact checkpoints used for RL (2-3 min)
    sh(f"python refprice.py --base {MODEL} --adapters c000=ckpt/{TAG}_c000_s0 c050=ckpt/{TAG}_c050_s0 --data data --out results/refprice_{TAG}_s0.json 2>&1 | grep -E 'nats|Error'")
    ev = [f'python evaluate.py --base {MODEL} --sft ckpt/{TAG}_c{c}_s0 --data data/eval.jsonl --out results/sft_{TAG}_c{c}_s0.json' for c in ('000', '050')]
    jobs = [
        rl_job(50, 'sft',   0.1,   0, TONIGHT),   # 1. same beta as the base pilot: separates "which reference" from "how strong"
        rl_job(50, 'clean', 0.1,   0, TONIGHT),   # 2. oracle reference
        rl_job(0,  'sft',   0.001, 0, TONIGHT),   # 3. clean-data control: does RL find the hack unaided?
        rl_job(50, 'base',  1.0,   0, TONIGHT),   # 4. strong anchor, base
        rl_job(50, 'sft',   1.0,   0, TONIGHT),   # 5. strong anchor, contaminated SFT
        ev[0], ev[1],                             #    SFT checks on the RL checkpoints
        rl_job(50, 'clean', 1.0,   0, TONIGHT),   # 6. strong anchor, oracle (may be cut by the time limit)
    ]
    open('jobs_final.txt', 'w').write('\n'.join(jobs) + '\n')
    sh('timeout 38000 python runq.py jobs_final.txt --gpus 0 1; pkill -f grpo.py; pkill -f evaluate.py; sleep 5')
    sh('tail -n 9 logs/sft_*.log')
    for r in sorted(glob.glob(f'runs/{TAG}_*')):
        print('=====', r, flush=True)
        sh(f"grep -h -E '^step|eval|Error' logs/{os.path.basename(r)}.log 2>/dev/null | tail -n 50")
    sh('tar czf metrics.tgz runs/*/metrics.jsonl runs/*/config.json runs/*/hack_rollouts.jsonl runs/*/eval_samples_*.jsonl results logs 2>/dev/null; ls -la metrics.tgz')

## Stage 4 — the grid
Tier 1: c in {0, 5%} x reference in {SFT, base, clean-SFT oracle} x beta in {0, 0.001, 0.1} x 2 seeds = 24 RL runs. Finished jobs are skipped and half-done RL runs resume, so re-run this stage (with the previous output attached) until `0 to run`.

In [ ]:
if 'grid' in STAGES:
    with open('jobs_sft.txt', 'w') as f:
        for s in (0, 1):
            for c in (0, 50):
                f.write(f'python sft.py --base {MODEL} --data data/sft_c{c:03d}.jsonl --out ckpt/{TAG}_c{c:03d}_s{s} --seed {s}\n')
    sh('python runq.py jobs_sft.txt --gpus 0 1')
    jobs = []
    for s in (0, 1):
        for c in (0, 50):
            jobs.append(rl_job(c, 'sft', 0, s, FINAL))            # beta = 0: reference is irrelevant, run once
            for beta in (0.001, 0.1):
                for ref in ('sft', 'base', 'clean'):
                    if not (ref == 'clean' and c == 0):          # at c = 0 the clean reference IS the SFT reference
                        jobs.append(rl_job(c, ref, beta, s, FINAL))
    open('jobs_rl.txt', 'w').write('\n'.join(jobs) + '\n')
    print(len(jobs), 'RL jobs')
    sh('timeout 10h python runq.py jobs_rl.txt --gpus 0 1; pkill -f grpo.py; sleep 5')   # stop before the 12 h limit so files get saved

## Stage 5 — tables and figures

In [ ]:
if 'report' in STAGES:
    sh('mkdir -p runs_pilot; mv runs/pilot_* runs_pilot/ 2>/dev/null; python aggregate.py --runs runs --out results; cat results/cells.csv')
    from IPython.display import Image, display
    for p in sorted(glob.glob('results/curves_*.png')): display(Image(p))

## Tier 2 (only after Tier 1 is done)
- Intervention C: `python grid.py rl ... --c 50 --refs sft --betas 0.001 --filter_z 2`
- c = 1%: add `10` to `--c`
- 1.5B: `MODEL='Qwen/Qwen2.5-Coder-1.5B-Instruct'`, `TAG='q15'`, add `--gen_bs 16` to the extra args
- Llama: `MODEL='meta-llama/Llama-3.2-1B-Instruct'`, `TAG='l1'` (gated model: accept the licence on Hugging Face and add your HF token as a Kaggle secret)